# LIBRARY

In [ ]:
import gzip
import pickle
import ast
 
import numpy as np
import pandas as pd
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN

# PREPARATION

## LOAD DATASET

In [18]:
MOTIFS_CSV   = "../1.DATASET/motifs_discords.csv"
DATASET_PATH = "../1.DATASET/CMI_timeseries_personalized.pkl.gz"
CHANNELS     = ["X", "Y", "Z", "enmo", "anglez", "light"]
W            = 10
COLS_TO_DROP = ["battery_voltage", "timestamp", "quarter",
                "relative_date_PCIAT", "non-wear_flag", "weekday"]
 
# Set to True to also include discord windows in the feature vector
# Recommendation: start with False — motifs are more stable and interpretable
USE_DISCORDS = False

In [19]:
with gzip.open(DATASET_PATH, "rb") as f:
    CMI = pickle.load(f)

data_clean = [df.drop(columns=COLS_TO_DROP, errors="ignore") for df in CMI]
print(f"  {len(data_clean)} subjects loaded")


  4307 subjects loaded


## NORMALIZATION

In [20]:
signals = ["X", "Y", "Z", "enmo", "anglez", "light"]

In [21]:
def fit_global_scalers(df_list, signals):
    """Fit one RobustScaler per signal using TRAIN data only."""
    global_scalers = {}
    print("Fitting global scalers on train data...")
    for signal in signals:
        all_values = []
        for df in df_list:
            if signal not in df.columns:
                continue
            y = df[signal].values
            if np.std(y) < 0.0001 or np.ptp(y) == 0:
                continue
            all_values.append(y)
        if not all_values:
            continue
        population = np.concatenate(all_values).reshape(-1, 1)
        q25, q75   = np.percentile(population, [25, 75])
        iqr        = q75 - q25
        signal_std = np.std(population)
        if iqr > 1e-4:
            scaler = RobustScaler()
            scaler.fit(population)
            global_scalers[signal] = ("robust", scaler)
            print(f"  {signal:<18} → RobustScaler   (IQR={iqr:.4f})")
        elif signal_std > 1e-6:
            global_scalers[signal] = ("std_fallback", np.median(population), signal_std)
            print(f"  {signal:<18} → Std fallback   (IQR too small)")
        else:
            global_scalers[signal] = ("zero", None)
            print(f"  {signal:<18} → Zeroed")
    return global_scalers


In [22]:
def apply_global_scalers(df_list, global_scalers):
    """Apply pre-fitted scalers to a list of DataFrames."""
    scaled_list = []
    for df in df_list:
        target_df = df.copy()
        for signal, scaler_info in global_scalers.items():
            if signal not in target_df.columns:
                continue
            y    = target_df[signal].values.copy()
            kind = scaler_info[0]
            if kind == "robust":
                _, scaler = scaler_info
                y_scaled  = scaler.transform(y.reshape(-1, 1)).flatten()
            elif kind == "std_fallback":
                _, median, std = scaler_info
                y_scaled  = (y - median) / std
            else:
                y_scaled  = np.zeros_like(y)
            target_df[signal] = y_scaled
        scaled_list.append(target_df)
    return scaled_list


In [23]:
global_scalers = fit_global_scalers(data_clean, signals)

all_scaled = apply_global_scalers(
    data_clean,
    global_scalers
)

Fitting global scalers on train data...
  X                  → RobustScaler   (IQR=0.8339)
  Y                  → RobustScaler   (IQR=0.4161)
  Z                  → RobustScaler   (IQR=0.4321)
  enmo               → RobustScaler   (IQR=0.0455)
  anglez             → RobustScaler   (IQR=5.6445)
  light              → RobustScaler   (IQR=1.8506)


## LOAD MOTIF DATASET

In [24]:
print("Loading motifs CSV …")
motifs_df = pd.read_csv(MOTIFS_CSV)

# Parse stringified lists safely
for col in motifs_df.columns:
    if col != "sample_index":
        motifs_df[col] = motifs_df[col].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

Loading motifs CSV …


## SIGNAL RETRIEVAL

In [25]:
# For every subject, for every channel, pull the raw windows
# at each motif start index  →  store as fixed-length vectors

In [26]:
def extract_windows(ts: np.ndarray, indices: list[int], w: int) -> list[np.ndarray]:
    """Return list of length-w windows; skip any that go out of bounds."""
    windows = []
    for idx in indices:
        if idx + w <= len(ts):
            windows.append(ts[idx: idx + w])
    return windows

In [27]:
records = []
 
for _, row in motifs_df.iterrows():
    subj_idx = int(row["sample_index"])
    df_subj  = all_scaled[subj_idx]
 
    subj_features = {"sample_index": subj_idx}
 
    for ch in CHANNELS:
        if ch not in df_subj.columns:
            subj_features[f"{ch}_motif_mean"]   = np.zeros(W)
            subj_features[f"{ch}_discord_mean"] = np.zeros(W)
            continue
 
        ts           = df_subj[ch].values.astype(float)
        motif_idxs   = row[f"{ch}_motifs"]
        discord_idxs = row[f"{ch}_discords"]
 
        motif_wins   = extract_windows(ts, motif_idxs,   W)
        discord_wins = extract_windows(ts, discord_idxs, W)
 
        subj_features[f"{ch}_motif_mean"]   = (
            np.mean(motif_wins,   axis=0) if motif_wins   else np.zeros(W)
        )
        subj_features[f"{ch}_discord_mean"] = (
            np.mean(discord_wins, axis=0) if discord_wins else np.zeros(W)
        )
 
    records.append(subj_features)
 
print(f"  Built {len(records)} subject records")

  Built 4307 subject records


## BUILD 3D FEATURE MATRIX

In [28]:
if USE_DISCORDS:
    X_3d = np.array([
        np.hstack([
            np.stack([r[f"{ch}_motif_mean"]   for ch in CHANNELS], axis=1),  # (W, 6)
            np.stack([r[f"{ch}_discord_mean"] for ch in CHANNELS], axis=1),  # (W, 6)
        ])  # (W, 12)
        for r in records
    ])  # (n_subjects, W, n_channels * 2)
    print(f"Feature matrix shape (motifs + discords): {X_3d.shape}")
else:
    X_3d = np.array([
        np.stack([r[f"{ch}_motif_mean"] for ch in CHANNELS], axis=1)  # (W, 6)
        for r in records
    ])  # (n_subjects, W, n_channels)
    print(f"Feature matrix shape (motifs only): {X_3d.shape}")

Feature matrix shape (motifs only): (4307, 10, 6)


# CLUSTERING k-means

## precompute dtw distance matrix

In [29]:
print("Computing DTW distance matrix (this may take a while) ...")
sakoe_radius = max(1, int(0.05 * W))
dist_mat = cdist_dtw(
    X_3d,
    global_constraint="sakoe_chiba",
    sakoe_chiba_radius=sakoe_radius
)
print(f"  Distance matrix shape: {dist_mat.shape}")

Computing DTW distance matrix (this may take a while) ...
  Distance matrix shape: (4307, 4307)


## elbow

In [ ]:
k_range  = range(2, 17)
sse_list = []
sil_list = []

print("Sweeping k ...")
for k in k_range:
    print(f"  Calculating {k}-means ...")
    km = TimeSeriesKMeans(
    n_clusters=k,
    metric="dtw",
    metric_params={"global_constraint": "sakoe_chiba", "sakoe_chiba_radius": sakoe_radius},
    n_init=1,
    max_iter=500,
    random_state=42,
    n_jobs=-1      # moved directly into the constructor
    )
    #km.set_config(**{"backend:parallel": "loky", "n_jobs": -1})
    km.fit(X_3d)
    sse_list.append(km.inertia_)
    sil_list.append(silhouette_score(dist_mat, km.labels_, metric="precomputed"))
    print(f"    inertia={km.inertia_:,.2f}  silhouette={sil_list[-1]:.4f}")

Sweeping k ...
  Calculating 2-means ...
    inertia=88.69  silhouette=0.9843
  Calculating 3-means ...


In [ ]:
# ──  Elbow + Silhouette ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), sse_list, "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (SSE)")
axes[0].set_title("Elbow curve")

axes[1].plot(list(k_range), sil_list, "s-", color="orange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score (DTW)")
axes[1].set_title("Silhouette scores")
# mark automatic best for reference, but you can override
auto_best_k = list(k_range)[int(np.argmax(sil_list))]
axes[1].axvline(auto_best_k, ls="--", color="red", label=f"auto best k={auto_best_k}")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_k = int(input(f"\nAuto best k={auto_best_k}. Enter k to use: "))

## final fit with best k

In [ ]:
km_final = TimeSeriesKMeans(
    n_clusters=best_k, metric="dtw",
    distance_params={"window": 0.05},
    n_init=5, max_iter=500, random_state=42
)
km_final.set_config(**{"backend:parallel": "loky", "n_jobs": -1})
km_final.fit(X_3d)
cluster_labels = km_final.labels_

## cluster analysis

In [ ]:
print("\n" + "="*50)
print("CLUSTER ANALYSIS")
print("="*50)
 
# — 9a. Cluster sizes
print("\n--- Cluster sizes ---")
cluster_counts = motifs_df["cluster"].value_counts().sort_index()
for c, n in cluster_counts.items():
    pct = 100 * n / len(motifs_df)
    print(f"  Cluster {c}: {n:4d} subjects  ({pct:.1f}%)")
 
# — 9b. Subject indices per cluster
print("\n--- Subject indices per cluster (first 10 shown) ---")
for c in range(best_k):
    members = motifs_df.loc[motifs_df["cluster"] == c, "sample_index"].tolist()
    print(f"  Cluster {c} ({len(members)} subjects): {members[:10]}{'...' if len(members) > 10 else ''}")
 
# — 9c. Centroids  (shape: best_k, W, n_channels)
#        km_final.cluster_centers_ is already the mean motif per cluster
centroids = km_final.cluster_centers_   # (best_k, W, n_channels)
print(f"\n--- Centroid shapes: {centroids.shape}  (n_clusters, W, n_channels) ---")
 
# — 9d. Per-cluster centroid stats across channels
print("\n--- Centroid summary (mean ± std across time steps, per channel) ---")
for c in range(best_k):
    print(f"\n  Cluster {c}:")
    for i, ch in enumerate(CHANNELS):
        vals = centroids[c, :, i]
        print(f"    {ch:>8s}:  mean={vals.mean():.4f}  std={vals.std():.4f}"
              f"  min={vals.min():.4f}  max={vals.max():.4f}")
 
# — 9e. Save cluster membership table
membership_df = motifs_df[["sample_index", "cluster"]].copy()
#membership_df.to_csv("subject_clusters.csv", index=False)
#print("\nCluster assignments saved → subject_clusters.csv")
 
# — 9f. Save centroid arrays
#np.save("cluster_centroids.npy", centroids)
#print("Centroids saved → cluster_centroids.npy")

## visualise

In [ ]:
X_flat = X_3d.reshape(len(X_3d), -1)   # flatten only for PCA / t-SNE
CMAP   = "tab10"

In [ ]:
 
# ── 10b. Cluster size bar chart ───────────────
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(cluster_counts.index, cluster_counts.values,
              color=plt.cm.tab10(np.linspace(0, 1, best_k)))
ax.bar_label(bars, fmt="%d")
ax.set_xlabel("Cluster"); ax.set_ylabel("Number of subjects")
ax.set_title(f"Cluster sizes  (k={best_k})")
ax.set_xticks(range(best_k))
plt.tight_layout()
#plt.savefig("02_cluster_sizes.png", dpi=150)
plt.show()
#print("Saved → 02_cluster_sizes.png")

In [ ]:
# ── 10c. PCA scatter ─────────────────────────
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_flat)
 
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                c=cluster_labels, cmap=CMAP, s=15, alpha=0.7)
# mark centroids in PCA space
cents_flat  = centroids.reshape(best_k, -1)
cents_pca   = pca.transform(cents_flat)
ax.scatter(cents_pca[:, 0], cents_pca[:, 1],
           c=range(best_k), cmap=CMAP,
           s=200, marker="*", edgecolors="black", linewidths=0.8,
           label="Centroid", zorder=5)
for c in range(best_k):
    ax.annotate(str(c), cents_pca[c], fontsize=9, fontweight="bold",
                ha="center", va="bottom", xytext=(0, 6), textcoords="offset points")
ax.set_title(f"PCA projection — k={best_k}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.colorbar(sc, ax=ax, label="Cluster")
ax.legend()
plt.tight_layout()
#plt.savefig("03_pca_scatter.png", dpi=150)
plt.show()
#print("Saved → 03_pca_scatter.png")

In [ ]:
# ── 10d. t-SNE scatter ───────────────────────
print("Running t-SNE (this may take a minute) ...")
tsne   = TSNE(n_components=2, perplexity=30, random_state=42,
              metric="precomputed", init="random")
X_tsne = tsne.fit_transform(dist_mat)   # uses the real DTW distances
 
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                c=cluster_labels, cmap=CMAP, s=15, alpha=0.7)
# annotate cluster centres in t-SNE space (mean position of members)
for c in range(best_k):
    mask = cluster_labels == c
    cx, cy = X_tsne[mask, 0].mean(), X_tsne[mask, 1].mean()
    ax.annotate(f"C{c}\n(n={mask.sum()})", (cx, cy),
                fontsize=9, fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6))
ax.set_title(f"t-SNE projection (DTW distances) — k={best_k}")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
plt.colorbar(sc, ax=ax, label="Cluster")
plt.tight_layout()
plt.savefig("04_tsne_scatter.png", dpi=150)
plt.show()
print("Saved → 04_tsne_scatter.png")

In [ ]:
# ── 10e. Centroid motif shapes per channel ───
#   One figure per cluster, subplots = one per channel
for c in range(best_k):
    members_mask = cluster_labels == c
    n_members    = members_mask.sum()
 
    fig, axes = plt.subplots(2, 3, figsize=(14, 6))
    fig.suptitle(f"Cluster {c}  —  {n_members} subjects  "
                 f"({100*n_members/len(cluster_labels):.1f}%)", fontsize=13)
    axes = axes.flatten()
 
    for i, ch in enumerate(CHANNELS):
        centroid_ch = centroids[c, :, i]       # (W,)
        member_seqs = X_3d[members_mask, :, i] # (n_members, W)
 
        # light grey background = individual member motifs
        for seq in member_seqs:
            axes[i].plot(seq, color="grey", alpha=0.15, linewidth=0.8)
 
        # bold line = centroid
        axes[i].plot(centroid_ch, color="black", linewidth=2, label="Centroid")
        axes[i].set_title(ch)
        axes[i].set_xlabel("Time step")
        axes[i].legend(fontsize=7)
 
    plt.tight_layout()
    plt.savefig(f"05_centroid_cluster{c}.png", dpi=150)
    plt.show()
    print(f"Saved → 05_centroid_cluster{c}.png")

# DBSCAN

## k distance plot (choose k visually)

In [ ]:
MINPTS_FOR_EPS = 16
 
# Use the precomputed DTW dist_mat directly
sorted_dists = np.sort(dist_mat, axis=1)   # for each point, sorted distances to all others
kth_dists    = np.sort(sorted_dists[:, MINPTS_FOR_EPS])  # k-th nearest neighbour distance
 
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(kth_dists)
ax.set_xlabel("Points sorted by distance")
ax.set_ylabel(f"{MINPTS_FOR_EPS}-NN DTW distance")
ax.set_title("k-distance plot — look for the elbow to choose eps")
ax.grid(axis="y", linestyle="--")
plt.tight_layout()
#plt.savefig("06_dbscan_optimal_eps.png", dpi=150)
plt.show()
#print("Saved → 06_dbscan_optimal_eps.png")
 
EPS = float(input("\nEnter eps value from the elbow plot: "))

## sweep min_samples

In [ ]:
minpts_range = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
sil_list     = []
valid_minpts = []
 
print(f"\nSweeping min_samples with eps={EPS} ...")
for minpts in minpts_range:
    db = DBSCAN(eps=EPS, min_samples=minpts, metric="precomputed")
    db.fit(dist_mat)
 
    non_noise  = db.labels_ != -1
    n_clusters = len(set(db.labels_[non_noise]))
 
    if non_noise.sum() > 1 and n_clusters > 1:
        sil = silhouette_score(dist_mat[np.ix_(non_noise, non_noise)],
                               db.labels_[non_noise],
                               metric="precomputed")
        sil_list.append(sil)
        valid_minpts.append(minpts)
        print(f"  min_samples={minpts:5d}  clusters={n_clusters}  "
              f"noise={( ~non_noise).sum()}  silhouette={sil:.4f}")
    else:
        print(f"  min_samples={minpts:5d}  → no valid clusters, skipped")
 
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(valid_minpts, sil_list, "o-")
ax.set_xlabel("min_samples"); ax.set_ylabel("Silhouette score (DTW)")
ax.set_title(f"Silhouette vs min_samples  (eps={EPS})")
ax.set_xscale("log"); ax.grid(linestyle="--")
plt.tight_layout()
#plt.savefig("07_dbscan_silhouette_sweep.png", dpi=150)
plt.show()
#print("Saved → 07_dbscan_silhouette_sweep.png")

## final dbscan

In [ ]:
best_minpts = valid_minpts[int(np.argmax(sil_list))]
print(f"\nBest min_samples by silhouette: {best_minpts}")
 
db_final       = DBSCAN(eps=EPS, min_samples=best_minpts, metric="precomputed")
db_final.fit(dist_mat)
cluster_labels = db_final.labels_
 
non_noise_mask = cluster_labels != -1
n_noise        = (~non_noise_mask).sum()
unique_labels  = sorted(set(cluster_labels))
n_clusters     = len([l for l in unique_labels if l != -1])
 
sil_final = silhouette_score(
    dist_mat[np.ix_(non_noise_mask, non_noise_mask)],
    cluster_labels[non_noise_mask],
    metric="precomputed"
)
print(f"  Clusters found : {n_clusters}")
print(f"  Noise points   : {n_noise}  ({100*n_noise/len(cluster_labels):.1f}%)")
print(f"  Silhouette     : {sil_final:.4f}")
 
motifs_df["cluster_dbscan"] = cluster_labels

## cluster analysis

In [ ]:
print("\n" + "="*50)
print("CLUSTER ANALYSIS")
print("="*50)
 
print("\n--- Cluster sizes (label -1 = noise) ---")
cluster_counts = motifs_df["cluster_dbscan"].value_counts().sort_index()
for c, n in cluster_counts.items():
    tag = " [NOISE]" if c == -1 else ""
    print(f"  Cluster {c:>3d}{tag}: {n:4d} subjects  ({100*n/len(motifs_df):.1f}%)")
 
print("\n--- Subject indices per cluster (first 10 shown) ---")
for c in unique_labels:
    members = motifs_df.loc[motifs_df["cluster_dbscan"] == c, "sample_index"].tolist()
    tag = " [NOISE]" if c == -1 else ""
    print(f"  Cluster {c}{tag} ({len(members)} subjects): "
          f"{members[:10]}{'...' if len(members) > 10 else ''}")
 
# Pseudo-centroids: subject closest to the mean DTW distance in each cluster
print("\n--- Pseudo-centroids (most central subject per cluster) ---")
for c in unique_labels:
    if c == -1:
        continue
    mask       = np.where(cluster_labels == c)[0]
    sub_dist   = dist_mat[np.ix_(mask, mask)]
    mean_dists = sub_dist.mean(axis=1)
    central_i  = mask[np.argmin(mean_dists)]
    print(f"  Cluster {c}: most central subject = sample_index "
          f"{motifs_df.iloc[central_i]['sample_index']}  "
          f"(avg DTW dist to cluster = {mean_dists.min():.4f})")
 
motifs_df[["sample_index", "cluster_dbscan"]].to_csv(
    "subject_clusters_dbscan.csv", index=False)
print("\nCluster assignments saved → subject_clusters_dbscan.csv")

## visualise

In [ ]:
def make_colors(labels, cmap_name="Spectral"):
    unique  = sorted(set(labels))
    n_real  = len([l for l in unique if l != -1])
    cmap    = plt.cm.get_cmap(cmap_name, max(n_real, 1))
    ci      = 0
    color_map = {}
    for l in unique:
        color_map[l] = "lightgrey" if l == -1 else cmap(ci / max(n_real - 1, 1))
        if l != -1:
            ci += 1
    return np.array([color_map[l] for l in labels])
 
colors = make_colors(cluster_labels)
 
handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=("lightgrey" if l == -1
                                else plt.cm.Spectral(i / max(n_clusters - 1, 1))),
               markersize=7,
               label=("Noise" if l == -1 else f"Cluster {l}"))
    for i, l in enumerate(unique_labels)
]
 
# — PCA (flatten X_3d for projection only)
X_flat = X_3d.reshape(len(X_3d), -1)
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_flat)
 
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, s=15, alpha=0.7)
for c in unique_labels:
    if c == -1:
        continue
    mask = cluster_labels == c
    cx, cy = X_pca[mask, 0].mean(), X_pca[mask, 1].mean()
    ax.annotate(f"C{c}\n(n={mask.sum()})", (cx, cy), fontsize=8,
                fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
ax.set_title(f"PCA — DBSCAN  (eps={EPS}, min_samples={best_minpts})")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.legend(handles=handles, loc="upper left", title="Clusters", fontsize=7)
plt.tight_layout()
#plt.savefig("08_dbscan_pca.png", dpi=150)
plt.show()
#print("Saved → 08_dbscan_pca.png")
 
# — t-SNE (precomputed DTW distances, same as DTW script)
print("Running t-SNE ...")
tsne   = TSNE(n_components=2, perplexity=30, random_state=42,
              metric="precomputed", init="random")
X_tsne = tsne.fit_transform(dist_mat)
 
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, s=15, alpha=0.7)
for c in unique_labels:
    if c == -1:
        continue
    mask = cluster_labels == c
    cx, cy = X_tsne[mask, 0].mean(), X_tsne[mask, 1].mean()
    ax.annotate(f"C{c}\n(n={mask.sum()})", (cx, cy), fontsize=8,
                fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))
ax.set_title(f"t-SNE — DBSCAN  (eps={EPS}, min_samples={best_minpts})")
ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2")
ax.legend(handles=handles, loc="upper left", title="Clusters", fontsize=7)
plt.tight_layout()
#plt.savefig("09_dbscan_tsne.png", dpi=150)
plt.show()
#print("Saved → 09_dbscan_tsne.png")